# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR\^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print the dataset high-level description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Each entity is referenced by its `@id`. This will help identify how the data is organized in the dataset and what you can analyze.

In [ ]:
# List all record sets in the dataset along with their @id, name, and description
record_sets_info = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        rec = {'@id': rs.id, 'name': getattr(rs, 'name', ''), 'description': getattr(rs, 'description', '')}
        record_sets_info.append(rec)
else:
    # Fallback: mlcroissant might use dataset.record_sets or dataset.metadata.record_sets
    try:
        for rs in dataset.record_sets:
            rec = {'@id': rs.id, 'name': getattr(rs, 'name', ''), 'description': getattr(rs, 'description', '')}
            record_sets_info.append(rec)
    except Exception:
        pass

if not record_sets_info:
    print("No record sets found in metadata.")
else:
    print("Available Record Sets:")
    for rec in record_sets_info:
        print(f"- @id: {rec['@id']} | name: {rec['name']} | description: {rec['description']}")

# Optionally, display available field information for a chosen record set
if record_sets_info:
    # Use the first record set as example
    example_record_set_id = record_sets_info[0]['@id']
    example_record_set = None
    for rs in metadata.record_sets:
        if rs.id == example_record_set_id:
            example_record_set = rs
    if example_record_set:
        print(f"\nFields for record set {example_record_set_id}:")
        for field in example_record_set.fields:
            print(f"  - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Prepare a list of all available record set @ids
record_set_ids = [rs['@id'] for rs in record_sets_info]
dataframes = {}

# Load each record set into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id} with {len(records)} rows.")
    else:
        print(f"Record set {record_set_id} has no records.")

if dataframes:
    # Use the first available DataFrame for demonstration
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"\nColumns in record set {main_record_set_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No DataFrames loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You must reference fields and columns by their `@id` values.

In [ ]:
# For demonstration, let's select a numeric field from the first record set
if dataframes:
    df = dataframes[main_record_set_id]

    # Try automatic detection of a numeric field (float/int)
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if not numeric_candidates:
        print("No numeric fields found in this record set.")
    else:
        numeric_field = numeric_candidates[0]  # e.g., '@id' of a numeric field (as per requirement)

        print(f"Analyzing numeric field: {numeric_field}")

        # Filter records by threshold > median value
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records in '{main_record_set_id}' where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"First 5 records: normalized {numeric_field}:\n")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical field (string, not unique)
        group_candidates = [c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() < len(df) // 2]
        if group_candidates:
            group_field = group_candidates[0]  # e.g., '@id' of a group field
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable string-based group field found for grouping.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize distributions or field relationships in the dataset.

This example plots the histogram of the numeric field used in the EDA above. Change `numeric_field` and `group_field` according to the specific dataset content and field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (by @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field exists, show a boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} distribution grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, you learned how to access a Croissant dataset package, enumerate and extract record sets, reference fields/columns by their `@id`, and perform exploratory data processing and visualization using `mlcroissant`.

**Key findings:**
- The dataset is rich in structured record sets and fields, each referenced by explicit `@id`s.
- You can dynamically extract and analyze any record set by referencing its `@id`.
- Standard EDA/visualization steps are supported, enabling reproducible analysis workflows.

For more details and to see each field's semantics, further explore the dataset's Croissant schema or read the [documentation](https://mlcommons.github.io/croissant/).